In [ ]:
import pandas as pd

In [ ]:
data_viz = pd.read_parquet('s3://dsp-ai-eval/racial_bias_org_change/rq1/outputs/openalex/data/visualization_data.parquet')

In [ ]:
data_viz.head()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

NESTA_COLOURS = [
    "#0000FF",
    "#FDB633",
    "#18A48C",
    "#9A1BBE",
    "#EB003B",
    "#FF6E47",
    "#646363",
    "#0F294A",
    "#97D9E3",
    "#A59BEE",
    "#F6A4B7",
    "#D2C9C0",
    # "#FFFFFF",
    "#000000",
]

fig = px.scatter(
        data_viz,
        x="x",
        y="y",
        # text="keywords",
        color="topic_name",
        hover_data={"topic_name": True, "title": True, "publication_year": True, "total_cites": True, "journal": True, "x": False, "y": False},
        color_discrete_sequence=NESTA_COLOURS,
        opacity=0.5,
    )

fig.update_layout(
        width=1200,  # Increase width
        height=800,  # Adjust height
        xaxis=dict(showticklabels=False, title_text=""),  # Hide x-axis ticks and title
        yaxis=dict(showticklabels=False, title_text=""),  # Hide y-axis ticks and title
        legend_title_text="",  # Hide legend title
        plot_bgcolor="white",  # Set background of the plot area to white
        paper_bgcolor="white",  # Set background of the entire figure to white
    )

fig.write_html('topic_scatterplot_rq1.html')

fig

# Identify abstracts that mention specific types of evidence

In [ ]:
# Define the terms to search for
terms = [
    'RCT', 'trial', 'randomised control trial', 'observational study','observational data','cohort', 
    'evidence', 'meta-analysis', 'meta-analyses', 'systematic review', 'evaluation'
]

# Create a regex pattern, ensuring word boundaries where necessary
pattern = r'|'.join([fr'\b{term}\b' for term in terms])

# Create the 'mentions_evidence' column
data_viz['mentions_evidence_type'] = data_viz['title_abstract'].str.contains(pattern, case=False, regex=True)

data_viz['mentions_evidence_type'].value_counts()


In [ ]:
symbol_map = {True: 'circle', False: 'square'}

data_viz['symbol'] = data_viz['mentions_evidence_type'].map(symbol_map)

In [ ]:

fig = px.scatter(
    data_viz,
    x="x",
    y="y",
    color="topic_name",
    symbol="mentions_evidence_type",  # Different shapes for mentions_evidence
    hover_data={
        "topic_name": True,
        "title": True,
        "publication_year": True,
        "total_cites": True,
        "journal": True,
        "x": False,
        "y": False
    },
    color_discrete_sequence=NESTA_COLOURS,
    opacity=0.3,
)

fig.update_layout(
    width=1200,
    height=800,
    xaxis=dict(showticklabels=False, title_text=""),
    yaxis=dict(showticklabels=False, title_text=""),
    legend_title_text="",
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()


# Identifying which search terms contribute to which topics

In [ ]:
crosstab_result = pd.crosstab(data_viz['search_term'], data_viz['topic_name'])
crosstab_result

In [ ]:
crosstab_result.to_csv('search_term_topic_crosstab.csv')